# RetinaGuard APTOS evaluation

This notebook creates a fixed stratified split, calibrates the abstention threshold on validation data, and reports locked test-set results against two baselines. It does not call OpenRouter because referral text is not part of the vision-model metrics.

Before running, accept the APTOS 2019 competition rules on Kaggle and add a Colab secret named `KAGGLE_API_TOKEN`.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo = Path('/content/PE6201-End-of-Course-Project')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/MauriceChang1027/PE6201-End-of-Course-Project', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements-colab.txt')], check=True)

In [ ]:
import kagglehub

data_dir = Path('/content/aptos2019')
competition_path = Path(kagglehub.competition_download(
    'aptos2019-blindness-detection',
    output_dir=str(data_dir),
))
competition_path

In [ ]:
labels_csv = next(data_dir.rglob('train.csv'))
images_dir = next(path for path in data_dir.rglob('train_images') if path.is_dir())
results_dir = repo / 'evaluation' / 'results'
print('Labels:', labels_csv)
print('Images:', images_dir)

In [ ]:
subprocess.run([
    sys.executable, '-m', 'scripts.evaluate_aptos',
    '--labels-csv', str(labels_csv),
    '--images-dir', str(images_dir),
    '--output-dir', str(results_dir),
    '--batch-size', '32',
    '--seed', '6201',
    '--target-sensitivity', '0.90',
    '--max-abstention-rate', '0.15',
], cwd=repo, check=True)

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv(results_dir / 'summary_metrics.csv'))
display(Image(filename=results_dir / 'confusion_matrix_referable_raw.png'))
display(Image(filename=results_dir / 'threshold_calibration.png'))
display(Image(filename=results_dir / 'baseline_comparison.png'))

In [ ]:
from google.colab import files
import shutil

archive = shutil.make_archive('/content/retinaguard_evaluation_results', 'zip', results_dir)
files.download(archive)